In [1]:
!pip install tensorflow==2.20.0 opencv-python matplotlib

Access is denied.


In [2]:
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt
import cv2

In [3]:
interpreter = tf.lite.Interpreter(model_path='3.tflite')
interpreter.allocate_tensors()

C:\Users\Yuvraj Malik\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [4]:
def calculate_angle(a, b, c):
    """
    Calculate angle between three points (in degrees).
    a, b, c: (x,y) coordinates
    Angle is at point b formed by vectors ab and cb
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ab = a - b
    cb = c - b

    # Cosine formula
    cosine_angle = np.dot(ab, cb) / (np.linalg.norm(ab) * np.linalg.norm(cb) + 1e-7)
    angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))

    return np.degrees(angle)


In [12]:
import numpy as np
import cv2
import math
import tensorflow as tf

# ---------------- GLOBAL VARS ----------------
MIN_CONF = 0.2
last_angles = {"L": None, "R": None}
stage = None   # down / up
counter = 0

# scoring
current_score = 0
rep_scores = []
min_angle_in_rep = None  # track lowest angle during down phase

# instructions overlay (NEW)
SHOW_INSTRUCTIONS = True              # toggle on/off
LANG = "both"                         # "en", "hi", or "both"
HINDI_FONT_PATH = None                # e.g. "NotoSansDevanagari-Regular.ttf" to render Hindi script

# ---------------- FUNCTIONS ----------------
def calculate_angle(a, b, c):
    """Calculate angle between three points (in degrees)."""
    A, B, C = np.array(a), np.array(b), np.array(c)
    BA, BC = A - B, C - B
    denom = np.linalg.norm(BA) * np.linalg.norm(BC)
    if denom == 0:
        return None
    cosang = np.dot(BA, BC) / denom
    return math.degrees(math.acos(np.clip(cosang, -1.0, 1.0)))


def get_leg_angle(name, hip, knee, ankle):
    """Calculate + smooth leg angle for one side."""
    global last_angles
    angle = None
    if hip[2] > MIN_CONF and knee[2] > MIN_CONF and ankle[2] > MIN_CONF:
        raw_angle = calculate_angle(hip[:2], knee[:2], ankle[:2])
        if last_angles[name] is not None:
            angle = 0.7 * last_angles[name] + 0.3 * raw_angle
        else:
            angle = raw_angle
        last_angles[name] = angle
    return angle


def draw_line(frame, p1, p2, color=(0, 0, 255)):
    """Draw a line if both points are confident."""
    if p1[2] > MIN_CONF and p2[2] > MIN_CONF:
        cv2.line(frame, p1[:2], p2[:2], color, 2)

# ---------- Instructions overlay (UPDATED: right-side panel) ----------
def draw_instructions(frame, lang="both"):
    h, w = frame.shape[:2]

    # add a panel area to the RIGHT of the camera feed
    extra_width = 400
    new_frame = np.zeros((h, w + extra_width, 3), dtype=np.uint8)
    new_frame[:, :w] = frame

    # translucent panel background
    overlay = new_frame.copy()
    cv2.rectangle(overlay, (w + 10, 10), (w + extra_width - 10, h - 10), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.35, new_frame, 0.65, 0, new_frame)

    # text positions
    x0 = w + 20
    y = 40
    line_h = 28

    # English lines
    lines_en = [
        "SQUAT INSTRUCTIONS",
        "Stance: feet shoulder-width",
        "Keep chest up, core tight",
        "Knees track toes (no cave-in)",
        "Depth: go 70 to 90 degree (ATG or parallel)",
        "Drive up through heels",
        "Breathing: inhale down, exhale up"
    ]

    for i, line in enumerate(lines_en):
        thickness = 2 if i == 0 else 1
        scale = 0.75 if i == 0 else 0.6
        cv2.putText(new_frame, line, (x0, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                    (255, 255, 255), thickness, cv2.LINE_AA)
        y += line_h

    # Hindi transliteration in English letters
    if lang in ("hi", "both"):
        lines_hi_rom = [
            
            "HINDI:",
            "Stance: pair kandhe-chodai par rakhein",
            "Seena upar, core tight rakhein",
            "Ghutaney panjon ki disha mein rakhein",
            "Gahrayi: 70 to 90 degree tak neeche jayein",
            "Aedi se bal dekar uthen",
            "Saans: neeche lete, upar chhodte jayein"
        ]
        for line in lines_hi_rom:
            cv2.putText(new_frame, line, (x0, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
            y += line_h

    return new_frame



# ---------------- VIDEO CAPTURE ----------------
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    # -------- Preprocess frame --------
    img = tf.image.resize_with_pad(np.expand_dims(frame.copy(), axis=0), 192, 192)
    input_image = tf.cast(img, dtype=tf.float32)

    # -------- Run inference --------
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], np.array(input_image))
    interpreter.invoke()
    keypoints_with_scores = interpreter.get_tensor(output_details[0]['index'])

    # -------- Render skeleton --------
    draw_connections(frame, keypoints_with_scores, EDGES, 0.4)
    draw_keypoints(frame, keypoints_with_scores, 0.4)

    # -------- Extract scaled keypoints --------
    shaped = np.squeeze(np.multiply(keypoints_with_scores, [h, w, 1]))
    # Left leg
    LHIP   = (int(shaped[11][1]), int(shaped[11][0]), shaped[11][2])
    LKNEE  = (int(shaped[13][1]), int(shaped[13][0]), shaped[13][2])
    LANK   = (int(shaped[15][1]), int(shaped[15][0]), shaped[15][2])
    # Right leg
    RHIP   = (int(shaped[12][1]), int(shaped[12][0]), shaped[12][2])
    RKNEE  = (int(shaped[14][1]), int(shaped[14][0]), shaped[14][2])
    RANK   = (int(shaped[16][1]), int(shaped[16][0]), shaped[16][2])

    # -------- Angles for both legs --------
    L_angle = get_leg_angle("L", LHIP, LKNEE, LANK)
    R_angle = get_leg_angle("R", RHIP, RKNEE, RANK)

    # -------- Double-leg squat logic with scoring --------
    if L_angle is not None and R_angle is not None:
        # Both legs must go down
        if L_angle < 100 and R_angle < 100:
            if stage != "down":
                stage = "down"
                min_angle_in_rep = min(L_angle, R_angle)
            else:
                # keep the lowest (deepest) angle reached
                if min_angle_in_rep is None:
                    min_angle_in_rep = min(L_angle, R_angle)
                else:
                    min_angle_in_rep = min(min_angle_in_rep, L_angle, R_angle)

        # Both legs must return up
        if L_angle > 150 and R_angle > 150 and stage == "down":
            stage = "up"
            counter += 1
            # score this rep (based on min_angle_in_rep)
            if min_angle_in_rep is not None:
                if 85 <= min_angle_in_rep <= 115:
                    rep_score = 100  # perfect depth (parallel)
                elif 80 <= min_angle_in_rep < 85 or 115 < min_angle_in_rep <= 130:
                    # still inside green range but not ideal
                    rep_score = 90
                elif 70 <= min_angle_in_rep < 80 or 130 < min_angle_in_rep <= 140:
                    # acceptable but shallow or very deep
                    rep_score = 75
                else:
                    # extreme shallow (>140) or extreme deep (<70)
                    rep_score = 50

                rep_scores.append(rep_score)
                current_score = int(rep_score)
            else:
                current_score = 0
            min_angle_in_rep = None



    # -------- Choose color for feedback --------
    color = (0, 0, 255)  # red by default
    if L_angle is not None and R_angle is not None:
        if (80 <= L_angle <= 130 and 80 <= R_angle <= 130) or \
           (150 <= L_angle <= 180 and 160 <= R_angle <= 180):
            color = (0, 255, 0)

    # -------- Draw skeleton legs with feedback color --------
    draw_line(frame, LHIP, LKNEE, color)
    draw_line(frame, LKNEE, LANK, color)
    draw_line(frame, RHIP, RKNEE, color)
    draw_line(frame, RKNEE, RANK, color)

    # -------- Display info --------
    if L_angle is not None:
        cv2.putText(frame, f"L: {int(L_angle)}°", (30, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    if R_angle is not None:
        cv2.putText(frame, f"R: {int(R_angle)}°", (30, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv2.putText(frame, f"Squats: {counter}", (30, 200),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)

    # always show score
    cv2.putText(frame, f"Last Score: {current_score}", (30, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    avg_score = int(sum(rep_scores) / len(rep_scores)) if rep_scores else 0
    cv2.putText(frame, f"Avg Score: {avg_score}", (30, 270),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)

    # -------- Instructions overlay call (UPDATED to use returned frame) --------
    if SHOW_INSTRUCTIONS:
        frame = draw_instructions(frame, lang=LANG)

    # -------- Show frame --------
    cv2.imshow('Movenet Lightning - Squats', frame)
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [6]:
shaped = np.squeeze(np.multiply(interpreter.get_tensor(interpreter.get_output_details()[0]['index']), [480,640,1]))

In [7]:
for kp in shaped:
    ky, kx, kp_conf = kp
    print(int(ky), int(kx), kp_conf)

281 289 0.571806788444519
246 312 0.7255728244781494
262 251 0.5244548320770264
235 336 0.560044527053833
263 201 0.5918360948562622
368 470 0.6824367046356201
373 141 0.6978816986083984
409 566 0.06151152774691582
459 97 0.06476857513189316
331 399 0.040199071168899536
347 84 0.09786276519298553
462 488 0.00426872493699193
199 174 0.015161413699388504
419 576 0.0077342502772808075
404 141 0.022151721641421318
371 570 0.0062081459909677505
346 83 0.012764491140842438


In [8]:
def draw_keypoints(frame, keypoints, confidence_threshold):
    y, x, c = frame.shape
    shaped = np.squeeze(np.multiply(keypoints, [y,x,1]))
    
    for kp in shaped:
        ky, kx, kp_conf = kp
        if kp_conf > confidence_threshold:
            cv2.circle(frame, (int(kx), int(ky)), 4, (0,255,0), -1) 

In [9]:
EDGES = {
    (0, 1): 'm',
    (0, 2): 'c',
    (1, 3): 'm',
    (2, 4): 'c',
    (0, 5): 'm',
    (0, 6): 'c',
    (5, 7): 'm',
    (7, 9): 'm',
    (6, 8): 'c',
    (8, 10): 'c',
    (5, 6): 'y',
    (5, 11): 'm',
    (6, 12): 'c',
    (11, 12): 'y',
    (11, 13): 'm',
    (13, 15): 'm',
    (12, 14): 'c',
    (14, 16): 'c'
}

In [10]:
for edge, color in EDGES.items():
    p1, p2 = edge
    y1, x1, c1 = shaped[p1]
    y2, x2, c2 = shaped[p2]
    print((int(x2), int(y2)))

(312, 246)
(251, 262)
(336, 235)
(201, 263)
(470, 368)
(141, 373)
(566, 409)
(399, 331)
(97, 459)
(84, 347)
(141, 373)
(488, 462)
(174, 199)
(174, 199)
(576, 419)
(570, 371)
(141, 404)
(83, 346)


In [11]:
def draw_connections(frame, keypoints, edges, confidence_threshold):
    y, x, c = frame.shape
    shaped = np.squeeze(np.multiply(keypoints, [y,x,1]))
    
    for edge, color in edges.items():
        p1, p2 = edge
        y1, x1, c1 = shaped[p1]
        y2, x2, c2 = shaped[p2]
        
        if (c1 > confidence_threshold) & (c2 > confidence_threshold):      
            cv2.line(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0,0,255), 2)